# [1장 2강] 실습: 데이터 분석 환경 구축과 EDA/데이터 전처리

## 실습 목표

- pandas와 numpy를 활용해 보험 데이터를 불러오고 기본 구조를 확인할 수 있다.
- 보험료와 주요 Feature의 분포 및 관계를 확인하고 해석할 수 있다.
- 중복 데이터, 결측치와 이상치 후보를 확인하고 처리할 수 있다.
- 수치형 Feature에 Standardization과 Min-Max Scaling을 적용할 수 있다.
- 범주형 Feature에 One-Hot Encoding을 적용할 수 있다.
- 전처리한 Feature와 Label을 결합하여 학습용 데이터를 만들 수 있다.

## 진행 방식

- Google Colab 또는 Jupyter Notebook에서 진행합니다.
- 각 문제의 설명과 요구사항을 확인한 뒤 코드 셀을 작성합니다.
- 먼저 스스로 해결한 뒤 실행 결과를 해석해봅니다.

## 사용 데이터

- 파일: `insurance(2).csv`
- 데이터 크기: 1,338행 × 7열
- Feature: `age`, `sex`, `bmi`, `children`, `smoker`, `region`
- Label: `charges`

| 컬럼 | 의미 |
|---|---|
| `age` | 가입자 나이 |
| `sex` | 성별 |
| `bmi` | 체질량지수 |
| `children` | 자녀 수 |
| `smoker` | 흡연 여부 |
| `region` | 거주 지역 |
| `charges` | 개인 의료보험료(예측 대상) |

## 실습 준비: 라이브러리 및 데이터 불러오기

아래 코드는 문제 풀이 전 공통으로 실행합니다. Google Colab을 사용한다면 `insurance(2).csv`를 Google Drive의 `MyDrive`에 저장합니다. Jupyter Notebook에서는 노트북과 같은 폴더에 파일을 저장합니다.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
    file_path = '/content/drive/MyDrive/insurance(2).csv'
except ModuleNotFoundError:
    file_path = 'insurance.csv'

insurance_df = pd.read_csv(file_path)

display(insurance_df.head())
print('데이터 크기:', insurance_df.shape)
insurance_df.info()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


데이터 크기: (1338, 7)
<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 73.3 KB


## 필수 1: 보험 데이터 EDA

### 문제 1-1: 주요 데이터 분포와 그룹별 보험료 확인하기

#### 문제 설명

보험사는 가입자의 기본 특성과 의료보험료 분포를 파악하려고 합니다. 수치형 Feature의 기초 통계를 확인하고, 흡연 여부와 지역에 따라 평균 보험료가 어떻게 다른지 비교합니다.

#### 요구사항

1. `sex`, `smoker`, `region`의 값별 개수를 출력합니다.
2. `age`, `bmi`, `children`, `charges`의 평균, 중앙값, 최솟값, 최댓값을 표로 출력합니다.
3. `smoker`별 평균 `charges`를 출력합니다.
4. `region`별 평균 `charges`를 내림차순으로 출력합니다.
5. 결과를 보고 보험료와 관련하여 가장 뚜렷하게 나타나는 특징을 한두 문장으로 설명합니다.

#### 출력 결과

- 성별, 흡연 여부, 지역별 데이터 개수
- 주요 수치형 컬럼의 평균·중앙값·최솟값·최댓값
- 흡연 여부별 평균 보험료
- 지역별 평균 보험료

#### 결과 해석 작성

> 결과를 보고 보험료와 관련하여 가장 뚜렷하게 나타나는 특징은 무엇인가요?

In [8]:
print("sex 값별개수:", insurance_df['sex'].value_counts().to_dict()) 
print("smoker 값별개수:", insurance_df['smoker'].value_counts().to_dict()) 
print("region 값별개수:", insurance_df['region'].value_counts().to_dict())

summary_df = insurance_df[['age', 'bmi', 'charges', 'children']].describe().T[
    ['mean', '50%', 'min', 'max']
]

summary_df.columns = ['평균', '중앙값', '최솟값', '최댓값']

display(summary_df)

sex 값별개수: {'male': 676, 'female': 662}
smoker 값별개수: {'no': 1064, 'yes': 274}
region 값별개수: {'southeast': 364, 'southwest': 325, 'northwest': 325, 'northeast': 324}


,평균,중앙값,최솟값,최댓값
age,39.207025,39.000,18.0000,64.00000
bmi,30.663397,30.400,15.9600,53.13000
charges,13270.422265,9382.033,1121.8739,63770.42801
children,1.094918,1.000,0.0000,5.00000


### 문제 1-2: 시각화로 보험료 분포와 관계 확인하기

#### 문제 설명

통계값만으로는 보험료의 치우침과 Feature 간 차이를 한눈에 파악하기 어렵습니다. 히스토그램과 막대그래프를 사용하여 보험료 분포와 흡연 여부에 따른 평균 보험료 차이를 확인합니다.

#### 요구사항

1. `charges`의 분포를 히스토그램으로 출력합니다.
2. `smoker`별 평균 `charges`를 막대그래프로 출력합니다.
3. 두 그래프에서 확인한 특징을 설명합니다.

#### 출력 결과

- 보험료 구간별 빈도를 나타낸 히스토그램
- 흡연 여부별 평균 보험료 막대그래프

#### 결과 해석 작성

> 보험료 분포는 어떤 형태이며, 흡연 여부에 따른 평균 보험료에는 어떤 차이가 있나요?

## 필수 2: 중복 데이터와 결측치 처리

### 문제 2-1: 데이터 품질 확인 및 처리하기

#### 문제 설명

모델 학습 전에 중복 데이터와 결측치를 확인해야 합니다. 중복 행은 동일한 관측값이 여러 번 반영되게 만들 수 있고, 결측치는 일부 분석과 모델 학습을 어렵게 만들 수 있습니다.

#### 요구사항

1. 전체 중복 행의 개수를 확인합니다.
2. 중복 행을 제거합니다.
3. 컬럼별 결측치 개수를 출력합니다.
4. 결측치가 존재하는 컬럼이 있는지 확인합니다.
5. 처리 후 중복 행 수와 전체 결측치 수를 출력합니다.
6. 이 데이터에서 결측치 대체가 필요한지 설명합니다.

#### 출력 결과

- 중복 제거 전 중복 행 수: `1`
- 처리 후 중복 행 수: `0`
- 모든 컬럼의 결측치 수: `0`

#### 결과 해석 작성

> 이 데이터에서는 결측치를 평균, 중앙값 또는 최빈값으로 대체해야 하나요? 그 이유는 무엇인가요?

## 필수 3: IQR 기반 이상치 후보 처리

### 문제 3-1: BMI 이상치 후보 확인 및 제거하기

#### 문제 설명

`bmi`에 일반적인 범위에서 크게 벗어난 값이 있는지 IQR 기준으로 확인합니다. IQR 범위를 벗어난 값은 오류로 단정할 수 없지만, 이번 실습에서는 이상치 탐지와 제거 과정을 연습하기 위해 범위 안의 데이터만 남깁니다.

#### 요구사항

1. `bmi`의 Q1, Q3와 IQR을 계산합니다.
2. 이상치 판단을 위한 하한값과 상한값을 계산합니다.
3. IQR 범위를 벗어난 이상치 후보의 개수를 출력합니다.
4. 이상치 후보의 `age`, `bmi`, `smoker`, `charges`를 출력합니다.
5. IQR 범위 안에 있는 데이터만 남기고 인덱스를 초기화합니다.
6. 이상치 처리 전후 행 수를 출력합니다.
7. IQR 이상치 후보를 항상 제거해도 되는지 설명합니다.

#### 출력 결과

- `bmi`의 Q1, Q3, IQR, 하한값, 상한값
- BMI 이상치 후보 개수와 목록
- 이상치 처리 전후 행 수

#### 결과 해석 작성

> IQR 범위를 벗어난 값을 항상 제거해도 되나요? 실제 분석에서는 무엇을 추가로 확인해야 하나요?

## 필수 4: 수치형 및 범주형 데이터 변환

### 문제 4-1: Scaling과 One-Hot Encoding 적용하기

#### 문제 설명

`age`, `bmi`, `children`은 값의 범위가 다르고, `sex`, `smoker`, `region`은 문자로 구성되어 있습니다. 수치형 Feature에는 두 가지 Scaling을 적용하고, 범주형 Feature에는 One-Hot Encoding을 적용합니다. 예측 대상인 `charges`는 Feature 변환 대상에서 제외합니다.

#### 요구사항

1. `age`, `bmi`, `children`에 StandardScaler를 적용합니다.
2. 같은 세 컬럼에 MinMaxScaler를 적용합니다.
3. `sex`, `smoker`, `region`에 One-Hot Encoding을 적용합니다.
4. 표준화, 정규화, 인코딩 결과의 앞 5행을 출력합니다.
5. Standardization과 Min-Max Scaling의 차이를 설명합니다.
6. `charges`를 Scaling 및 Encoding 대상에서 제외한 이유를 설명합니다.

#### 출력 결과

- `_standard`가 붙은 표준화 컬럼
- `_minmax`가 붙은 0~1 범위의 정규화 컬럼
- 범주가 0과 1로 변환된 One-Hot Encoding 컬럼

#### 결과 해석 작성

> Standardization과 Min-Max Scaling은 결과 범위가 어떻게 다른가요? 또한 `charges`는 왜 Feature 변환 대상에서 제외했나요?

## 심화 1: 최종 학습 데이터 만들기

### 문제 5-1: 전처리 데이터 결합 및 저장하기

#### 문제 설명

표준화된 수치형 Feature, One-Hot Encoding된 범주형 Feature와 보험료 Label을 하나의 DataFrame으로 결합합니다. Min-Max Scaling 결과는 비교용으로만 사용하고 최종 학습 데이터에는 Standardization 결과를 사용합니다.

#### 요구사항

1. `standard_df`와 `encoded_df`를 열 방향으로 결합합니다.
2. 원본 `charges`를 Label로 추가합니다.
3. 최종 데이터의 앞 5행, 크기와 전체 결측치 수를 출력합니다.
4. 결과를 `insurance_preprocessed.csv`로 저장합니다.
5. 최종 데이터에 원본 범주형 컬럼이 없는 이유를 설명합니다.

#### 출력 결과

- 표준화된 수치형 Feature
- One-Hot Encoding된 범주형 Feature
- Label인 `charges`
- 최종 데이터의 크기와 전체 결측치 수 `0`
- `insurance_preprocessed.csv` 파일 생성

#### 결과 해석 작성

> 최종 학습 데이터에 `sex`, `smoker`, `region` 원본 컬럼이 없고 여러 개의 0과 1 컬럼이 있는 이유는 무엇인가요?

## 실습 마무리

1. EDA와 전처리는 모델 학습 전에 왜 필요한가요?
2. IQR 범위를 벗어난 값은 왜 바로 오류라고 단정할 수 없나요?
3. 수치형 Feature와 범주형 Feature는 각각 어떤 방식으로 변환했나요?